<a href="https://colab.research.google.com/github/erdenebayrd/mit/blob/main/finetune_from_backbone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Tue Sep  8 11:32:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -la /content/drive/MyDrive/silent_speech
!echo "--- fine-tune runs ---" && ls -la /content/drive/MyDrive/silent_speech/output_finetune

total 10557442
-rw------- 1 root root 6866471136 Aug 24 03:45 emg_dataset.h5
-rw------- 1 root root 3919507637 Aug 24 03:45 emg_data.tar.gz
drwx------ 2 root root       4096 Aug 24 03:49 KenLM
drwx------ 2 root root       4096 Aug 24 03:50 output
drwx------ 2 root root       4096 Aug 31 05:44 output_finetune
-rw------- 1 root root   10513993 Aug 31 04:28 resume.pt
-rw------- 1 root root   14313646 Aug 31 05:44 TinyMyo_backbone.pt
--- fine-tune runs ---
total 107076
-rw------- 1 root root 18273881 Aug 31 05:46 model_20260831_054457_best.pt
-rw------- 1 root root 18273881 Aug 31 05:51 model_20260831_054457_last.pt
-rw------- 1 root root 18273881 Aug 31 06:07 model_20260831_060655_best.pt
-rw------- 1 root root 18273881 Aug 31 06:20 model_20260831_060655_last.pt
-rw------- 1 root root 18273881 Sep  7 02:25 model_20260907_022453_best.pt
-rw------- 1 root root 18273881 Sep  7 02:36 model_20260907_022453_last.pt


In [ ]:
%cd /content
!git clone https://github.com/MatteoFasulo/silent_speech.git
%cd /content/silent_speech
!git submodule update --init text_alignments
!tar -xzf text_alignments/text_alignments.tar.gz
!sed -i '/norm_layer=norm_layer,/d' architecture.py

/content
Cloning into 'silent_speech'...
remote: Enumerating objects: 518, done.
remote: Counting objects: 100% (362/362), done.
remote: Compressing objects: 100% (212/212), done.
remote: Total 518 (delta 210), reused 263 (delta 138), pack-reused 156 (from 1)
Receiving objects: 100% (518/518), 2.61 MiB | 6.40 MiB/s, done.
Resolving deltas: 100% (279/279), done.
/content/silent_speech
Submodule 'text_alignments' (https://github.com/dgaddy/silent_speech_alignments.git) registered for path 'text_alignments'
Cloning into '/content/silent_speech/text_alignments'...
Submodule path 'text_alignments': checked out '5c71ae9fcbb94e74e19eb9547c3b404baf6126a7'


In [ ]:
!pip install -q flashlight-text jiwer timm torchinfo torchprofile wandb tensorboard \
  librosa soundfile noisereduce resampy praat-textgrids unidecode \
  h5py scipy joblib matplotlib tqdm requests numpy huggingface_hub safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 125.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 141.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 163.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 132.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
%env DATA_PATH=/content/data

env: DATA_PATH=/content/data


In [ ]:
%cd /content/silent_speech
!mkdir -p /content/data/Gaddy/h5
!cp /content/drive/MyDrive/silent_speech/emg_dataset.h5 /content/data/Gaddy/h5/ && echo "h5 copied" || echo "!! h5 NOT on Drive"
!cp -r /content/drive/MyDrive/silent_speech/KenLM /content/silent_speech/ && echo "KenLM copied" || echo "!! KenLM NOT on Drive"
!cp /content/drive/MyDrive/silent_speech/emg_data.tar.gz /content/data/Gaddy/ 2>/dev/null && echo "tar restored" || echo "no tar on Drive; downloading fresh"
!python download_data.py

/content/silent_speech
h5 copied
KenLM copied
tar restored
2026-09-08 11:38:55.474365: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-08 11:38:55.491291: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788867535.512398    2914 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788867535.518862    2914 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-08 11:38:55.539706: I tensorflow/core/platform/cpu_feature_guard.cc:210] Thi

In [ ]:
import glob, os, re, json, torch
root = "/content/drive/MyDrive/silent_speech"
ft   = f"{root}/output_finetune"

# pick the model to continue from: your newest fine-tuned BEST checkpoint
cands = sorted(glob.glob(f"{ft}/model_*_best.pt"), key=os.path.getmtime)   # swap to "_last.pt" to continue from the very latest weights
if cands:
    src, mode, lr = cands[-1], "continue fine-tune", 2e-4
else:
    # no fine-tune found -> start fresh from the TinyMyo backbone (download if missing)
    src, mode, lr = f"{root}/TinyMyo_backbone.pt", "fresh fine-tune from backbone", 5e-4
    if not os.path.exists(src):
        from huggingface_hub import hf_hub_download
        from safetensors.torch import load_file
        sd = load_file(hf_hub_download("MatteoFasulo/TinyMyo", "pretraining/TinyMyo/TinyMyo.safetensors"))
        sd = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in sd.items()}
        torch.save({"state_dict": sd}, src)

# wrap for the loader + detect layer count (should be 8)
raw   = torch.load(src, map_location="cpu", weights_only=False)
state = raw["state_dict"] if isinstance(raw, dict) and "state_dict" in raw else raw
idx   = [int(re.match(r"blocks\.(\d+)\.", k).group(1)) for k in state if re.match(r"blocks\.(\d+)\.", k)]
n_layers = (max(idx) + 1) if idx else 8
torch.save({"state_dict": state}, f"{root}/resume_finetune.pt")

# config
p = "/content/silent_speech/config/recognition_model.json"
cfg = json.load(open(p))
cfg["num_layers"]          = n_layers
cfg["start_training_from"] = f"{root}/resume_finetune.pt"
cfg["ckpt_directory"]      = ft
cfg["num_epochs"]          = 60
cfg["eval_interval"]       = 5
cfg["num_workers"]         = os.cpu_count()
cfg["learning_rate"]       = lr
json.dump(cfg, open(p, "w"), indent=4)
print(f"mode       : {mode}\nfrom       : {src}\nnum_layers : {n_layers}\nlr         : {lr}\nsaving to  : {ft}")

mode       : continue fine-tune
from       : /content/drive/MyDrive/silent_speech/output_finetune/model_20260907_022453_best.pt
num_layers : 8
lr         : 0.0002
saving to  : /content/drive/MyDrive/silent_speech/output_finetune


In [ ]:
import os
def chk(l, path): print(f"{l:16}: {'OK' if os.path.exists(path) else 'MISSING <- fix this'}")
chk("h5",         "/content/data/Gaddy/h5/emg_dataset.h5")
chk("raw voiced", "/content/data/Gaddy/emg_data/voiced_parallel_data")
chk("KenLM lm",   "/content/silent_speech/KenLM/lm.bin")
chk("lexicon",    "/content/silent_speech/KenLM/gaddy_lexicon.txt")
chk("resume ckpt","/content/drive/MyDrive/silent_speech/resume_finetune.pt")

h5              : OK
raw voiced      : OK
KenLM lm        : OK
lexicon         : OK
resume ckpt     : OK


In [ ]:
import os, glob
os.chdir("/content/silent_speech")
os.environ["START"] = sorted(glob.glob("/content/drive/MyDrive/silent_speech/output_finetune/model_*_best.pt"), key=os.path.getmtime)[-1]
print("starting-point test WER for:", os.environ["START"])
!python recognition_model.py --evaluate_saved "$START"

starting-point test WER for: /content/drive/MyDrive/silent_speech/output_finetune/model_20260907_022453_best.pt
2026-09-08 11:41:08.129420: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-08 11:41:08.147722: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788867668.168755    3503 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788867668.175121    3503 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-08 11:41:08.196508: I t

In [ ]:
# %cd /content/silent_speech
# !python recognition_model.py

In [ ]:
# import os, glob
# os.chdir("/content/silent_speech")
# os.environ["BEST"] = sorted(glob.glob("/content/drive/MyDrive/silent_speech/output_finetune/model_*_best.pt"), key=os.path.getmtime)[-1]
# print("evaluating:", os.environ["BEST"])
# !python recognition_model.py --evaluate_saved "$BEST"

In [ ]:
import os, glob
os.chdir("/content/silent_speech")
for c in sorted(glob.glob("/content/drive/MyDrive/silent_speech/output_finetune/model_*_best.pt")):
    os.environ["CK"] = c
    print("\n=====", os.path.basename(c))
    !python recognition_model.py --evaluate_saved "$CK" 2>/dev/null | grep "^WER"


===== model_20260831_054457_best.pt
WER: 1.0

===== model_20260831_060655_best.pt
WER: 1.0

===== model_20260907_022453_best.pt
WER: 1.0


In [ ]:
import os, torch
os.chdir("/content/silent_speech")
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from architecture import EMGTransformer
from data_utils import TextTransform

sd = load_file(hf_hub_download("MatteoFasulo/TinyMyo", "pretraining/TinyMyo/TinyMyo.safetensors"))
clean = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in sd.items()}
m = EMGTransformer(num_features=8, num_outs=len(TextTransform().chars)+1, in_chans=8,
                   embed_dim=192, n_layer=8, n_head=3, mlp_ratio=4)
missing, unexpected = m.load_state_dict(clean, strict=False)
total_blk  = sum(1 for k in m.state_dict() if k.startswith("blocks."))
loaded_blk = sum(1 for k in m.state_dict() if k.startswith("blocks.") and k not in missing)
print(f"transformer-block tensors LOADED: {loaded_blk} / {total_blk}")
print("\nbackbone keys (first 10):")
for k, v in list(clean.items())[:10]: print("  ", k, tuple(v.shape))
print("\nmodel keys for block 0:")
for k in m.state_dict():
    if k.startswith("blocks.0."): print("  ", k)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pretraining/TinyMyo/TinyMyo.safetensors:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

transformer-block tensors LOADED: 96 / 104

backbone keys (first 10):
   blocks.0.attn.proj.bias (192,)
   blocks.0.attn.proj.weight (192, 192)
   blocks.0.attn.qkv.bias (576,)
   blocks.0.attn.qkv.weight (576, 192)
   blocks.0.mlp.fc1.bias (768,)
   blocks.0.mlp.fc1.weight (768, 192)
   blocks.0.mlp.fc2.bias (192,)
   blocks.0.mlp.fc2.weight (192, 768)
   blocks.0.norm1.bias (192,)
   blocks.0.norm1.weight (192,)

model keys for block 0:
   blocks.0.attn.qkv.weight
   blocks.0.attn.qkv.bias
   blocks.0.attn.proj.weight
   blocks.0.attn.proj.bias
   blocks.0.attn.relative_positional.embeddings
   blocks.0.norm1.weight
   blocks.0.norm1.bias
   blocks.0.mlp.fc1.weight
   blocks.0.mlp.fc1.bias
   blocks.0.mlp.fc2.weight
   blocks.0.mlp.fc2.bias
   blocks.0.norm2.weight
   blocks.0.norm2.bias


In [ ]:
import os, torch
os.chdir("/content/silent_speech")
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from architecture import EMGTransformer
from data_utils import TextTransform
root = "/content/drive/MyDrive/silent_speech"

sd = load_file(hf_hub_download("MatteoFasulo/TinyMyo", "pretraining/TinyMyo/TinyMyo.safetensors"))
full = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in sd.items()}
m = EMGTransformer(num_features=8, num_outs=len(TextTransform().chars)+1, in_chans=8,
                   embed_dim=192, n_layer=8, n_head=3, mlp_ratio=4)
# zero the 8 relative-position tables TinyMyo lacks -> transformer starts exactly as pretrained
for k, v in m.state_dict().items():
    if "relative_positional" in k and k not in full:
        full[k] = torch.zeros_like(v)
missing, unexpected = m.load_state_dict(full, strict=False)
print("block tensors loaded:", sum(1 for k in m.state_dict() if k.startswith("blocks.") and k not in missing), "/ 104")
print("trained fresh (front-end + head):", len(missing), "tensors")
torch.save({"state_dict": full}, f"{root}/TinyMyo_backbone.pt")
print("saved ->", f"{root}/TinyMyo_backbone.pt")

block tensors loaded: 104 / 104
trained fresh (front-end + head): 112 tensors
saved -> /content/drive/MyDrive/silent_speech/TinyMyo_backbone.pt


In [ ]:
import os, json
root = "/content/drive/MyDrive/silent_speech"
p = "/content/silent_speech/config/recognition_model.json"
cfg = json.load(open(p))
cfg.update({"num_layers": 8, "start_training_from": f"{root}/TinyMyo_backbone.pt", "freeze_blocks": True,
            "ckpt_directory": f"{root}/output_finetune_stage1", "num_epochs": 20,
            "eval_interval": 2, "num_workers": os.cpu_count(), "learning_rate": 5e-4})
json.dump(cfg, open(p, "w"), indent=4)
print("STAGE 1 ready: transformer frozen, training conv front-end + head")

STAGE 1 ready: transformer frozen, training conv front-end + head


In [ ]:
%cd /content/silent_speech
!python recognition_model.py

/content/silent_speech
2026-09-08 12:07:37.808056: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788869257.832001   10786 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788869257.838608   10786 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
['recognition_model.py']
output example: (/content/data/Gaddy/emg_data/silent_parallel_data/5-9_silent, '252')
train / dev split: 8399 30
Layer (type:depth-idx)                                            Output Shape              Param #
EMGTransformer                                                    [1, 200, 38]              --
├─Sequential: 1-1                                                 [1, 192, 200]  

In [ ]:
import os, glob, torch
os.chdir("/content/silent_speech")
root = "/content/drive/MyDrive/silent_speech"

# pretrained TinyMyo transformer (8 blocks, zeroed rel-pos) from the backbone we built
bb = torch.load(f"{root}/TinyMyo_backbone.pt", map_location="cpu", weights_only=False)["state_dict"]
hybrid = {k: v for k, v in bb.items() if k.startswith("blocks.")}

# your TRAINED conv front-end + head from the from-scratch model (the 40.8% one)
scratch_best = sorted(glob.glob(f"{root}/output/model_*_best.pt"), key=os.path.getmtime)[-1]
fs = torch.load(scratch_best, map_location="cpu", weights_only=False)
for k, v in fs.items():
    if k.startswith(("conv_blocks.", "w_raw_in.", "w_out.")):
        hybrid[k] = v

print("front-end/head from:", os.path.basename(scratch_best))
print("transformer tensors:", sum(k.startswith("blocks.") for k in hybrid),
      "| front-end+head tensors:", sum(not k.startswith("blocks.") for k in hybrid))
torch.save({"state_dict": hybrid}, f"{root}/hybrid_warmstart.pt")
print("saved ->", f"{root}/hybrid_warmstart.pt")

front-end/head from: model_20260831_044634_best.pt
transformer tensors: 104 | front-end+head tensors: 130
saved -> /content/drive/MyDrive/silent_speech/hybrid_warmstart.pt


In [ ]:
import os, json
root = "/content/drive/MyDrive/silent_speech"
p = "/content/silent_speech/config/recognition_model.json"
cfg = json.load(open(p))
cfg.update({"num_layers": 8, "start_training_from": f"{root}/hybrid_warmstart.pt", "freeze_blocks": False,
            "ckpt_directory": f"{root}/output_finetune_hybrid", "num_epochs": 60,
            "eval_interval": 2, "num_workers": os.cpu_count(), "learning_rate": 2e-4})
json.dump(cfg, open(p, "w"), indent=4)
print("HYBRID fine-tune ready: pretrained transformer + trained front-end, all trainable, lr 2e-4")

HYBRID fine-tune ready: pretrained transformer + trained front-end, all trainable, lr 2e-4


In [ ]:
%cd /content/silent_speech
!python recognition_model.py

/content/silent_speech
2026-09-08 12:18:57.724714: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788869937.745830   14165 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788869937.752148   14165 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
['recognition_model.py']
output example: (/content/data/Gaddy/emg_data/silent_parallel_data/5-9_silent, '308')
train / dev split: 8399 30
Layer (type:depth-idx)                                            Output Shape              Param #
EMGTransformer                                                    [1, 200, 38]              --
├─Sequential: 1-1                                                 [1, 192, 200]  

In [ ]:
import os, glob, json, torch
root = "/content/drive/MyDrive/silent_speech"
last = sorted(glob.glob(f"{root}/output_finetune_hybrid/model_*_last.pt"), key=os.path.getmtime)[-1]
torch.save({"state_dict": torch.load(last, map_location="cpu", weights_only=False)}, f"{root}/resume_hybrid.pt")
p = "/content/silent_speech/config/recognition_model.json"; cfg = json.load(open(p))
cfg["start_training_from"] = f"{root}/resume_hybrid.pt"; json.dump(cfg, open(p, "w"), indent=4)
print("resuming from", os.path.basename(last))

resuming from model_20260908_121901_last.pt


In [ ]:
%cd /content/silent_speech
!python recognition_model.py

/content/silent_speech
2026-09-08 12:29:28.671578: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788870568.693146   17168 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788870568.699540   17168 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
['recognition_model.py']
output example: (/content/data/Gaddy/emg_data/silent_parallel_data/5-11_silent, '10')
train / dev split: 8399 30
Layer (type:depth-idx)                                            Output Shape              Param #
EMGTransformer                                                    [1, 200, 38]              --
├─Sequential: 1-1                                                 [1, 192, 200]  